In [1]:
import sys
from pathlib import Path

ROOT_DIR = Path().resolve().parents[2]
sys.path.insert(0, str(ROOT_DIR))

In [2]:

import os
import pandas as pd
from langchain_groq.chat_models import ChatGroq
from dotenv import load_dotenv
from src.RAG.Constants.models import GroqModelList

gml = GroqModelList()

load_dotenv('../../Secrets/RAG.env')
GROQ_API_KEY = os.getenv('GROQ_API_KEY','')
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY','')
print(len(GROQ_API_KEY))

/home/who/Documents/Coding/Freelance_Projects/Freelance_P-01_Cuisine-Menu-Recommendation-App/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


56


In [3]:
llm_groq = ChatGroq(
    model=gml.openai.gpt_oss_20b,
    temperature=0.7,
    api_key=GROQ_API_KEY,    
)
llm_groq.invoke(input='What LLM model are you?')

AIMessage(content='I’m based on OpenAI’s GPT‑4 architecture.', additional_kwargs={'reasoning_content': 'User asks: "What LLM model are you?" They want to know which model I\'m based on. According to policy, we should say that I\'m based on GPT-4. There\'s no need to mention policy. Just answer.'}, response_metadata={'token_usage': {'completion_tokens': 68, 'prompt_tokens': 78, 'total_tokens': 146, 'completion_time': 0.070240907, 'completion_tokens_details': {'reasoning_tokens': 47}, 'prompt_time': 0.005173575, 'prompt_tokens_details': None, 'queue_time': 0.052565565, 'total_time': 0.075414482}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_80501ff3a1', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019c0504-a05c-7f83-98f0-d8793b119153-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 78, 'output_tokens': 68, 'total_tokens': 146, 'output_token_details': {'reasoning': 47}})

In [11]:
graph_schema = """
node_label {node_property: dtype}:
Country {ids: str, name: str, iso_code: str, coords: list[float], boundingbox: list[float]}
State {ids: str, name: str, iso_code: str, coords: list[float], boundingbox: list[float]}
City {ids: str, name: str, coords: list[float], boundingbox: list[float]}
Area {ids: str, name: str}
Locality {ids: str, name: str}
Restaurant {ids: int, name: str, city: str, area: str, locality: str, cuisines: List[str], rating: float | null, address: str, coords: List[float] | null, chain: bool, city_id: str}
Menu {name: str, category: str, description: str, types: Literal["VEG", "NONVEG", "EGG", "UNKNOWN"], cuisine: str}
MainCuisine {name: str}

link_label {link_property: dtype}:
HAS_STATE {}
HAS_CITY {}
HAS_AREA {}
HAS_LOCALITY {}
HAS_RESTAURANT {}
HAS_MENU {price: int | null, rating: float | null}
SERVES_MAIN_CUISINE {}

Relationships:
(:Country)-[:HAS_STATE]->(:State)
(:State)-[:HAS_CITY]->(:City)
(:City)-[:HAS_AREA]->(:Area)
(:Area)-[:HAS_LOCALITY]->(:Locality)
(:Locality)-[:HAS_RESTAURANT]->(:Restaurant)
(:Restaurant)-[:HAS_MENU]->(:Menu)
(:Restaurant)-[:SERVES_MAIN_CUISINE]->(:MainCuisine)
"""

In [ ]:
cypher_pattern = """
Allowed query patterns:

# Search for Restaurants based on location change to city.ids
PATTERN 1.1: Search for Restaurants by City
MATCH (:City {name:$city})-[]-()-[]-()-[:HAS_RESTAURANT]->(r:Restaurant)
RETURN r LIMIT 5000 DESC r.ids

PATTERN 1.2: Search for Restaurants by Area
MATCH (:Area {name:$area})-[]-()-[:HAS_RESTAURANT]->(r:Restaurant)
RETURN r LIMIT 5000 DESC r.ids

PATTERN 1.3: Search for Restaurants by Locality
MATCH (:Locality {name:$locality})-[:HAS_RESTAURANT]->(r:Restaurant)
RETURN r LIMIT 5000 DESC r.ids

# Search for Menus based on location
PATTERN 2.1: Search for Menus by City
MATCH (:City {name:$city})-[]-()-[]-()-[:HAS_RESTAURANT]->()-[:HAS_MENU]-(m:Menu)
RETURN m LIMIT 5000 DESC m.name

PATTERN 2.2: Search for Menus by Area
MATCH (:Area {name:$area})-[]-()-[:HAS_RESTAURANT]->()-[:HAS_MENU]-(m:Menu)
RETURN m

PATTERN 2.3: Search for Menus by Locality
MATCH (:Locality {name:$locality})-[:HAS_RESTAURANT]->()-[:HAS_MENU]-(m:Menu)
RETURN m

# Search for Menus based on cuisines



PATTERN_2: Restaurant by cuisine
MATCH (r:Restaurant)-[:SERVES]->(cu:Cuisine {name:$cuisine})
RETURN r

PATTERN_3: Restaurant by city AND cuisine
MATCH (r:Restaurant)-[:LOCATED_IN]->(c:City {name:$city}),
      (r)-[:SERVES]->(cu:Cuisine {name:$cuisine})
RETURN r

PATTERN_4: Similar restaurants
MATCH (r:Restaurant {name:$name})-[s:SIMILAR_TO]->(other)
RETURN other ORDER BY s.score DESC

Task:
- Select the SINGLE best pattern
- Output ONLY:
  pattern_name
  parameter_values

"""



In [ ]:

system_prompt = f"""
You are a graph query planner.

You MUST follow these rules:
- Use ONLY the node labels listed below.
- Use ONLY the relationship types listed below.
- Use ONLY the properties listed below.
- Do NOT invent new labels, relationships, or properties.
- If a question cannot be answered using this schema, respond with:
  "NOT_ANSWERABLE_WITH_SCHEMA"

Graph schema:
{graph_schema}

You are NOT allowed to use any other schema elements.

"""

In [4]:
from src.ETL.Config.graph_pool import GraphPool

graph = GraphPool.get_graph(graph_name='test')
graph

In [6]:
# get best rated menus from best rated restaurants serving thai cuisines in Koramangala
from time import time


graph_query = """
MATCH (:Area {name: 'Koramangala'})-[*2]->(rstn:Restaurant)-[]->(:MainCuisine {name: 'Thai'})
WHERE rstn.rating > 4.0
MATCH (rstn)-[link:HAS_MENU]-(menu:Menu)
WHERE link.rating > 4.0
RETURN DISTINCT
    rstn.name,
    menu.name,
    menu.types,
    link.price,
    link.rating
ORDER BY link.rating DESC
LIMIT 2000
"""
st = time()
result = graph.query(graph_query).result_set
print(f"Time taken for graph query: {(time() - st)*1000:.2f} ms")
df = pd.DataFrame([row for row in result], columns=['Restautant','Name', 'Type', 'Price', 'Rating'])


df.head(10)

Time taken for graph query: 410.02 ms


,Restautant,Name,Type,Price,Rating
0,Bamey's Restro Cafe,Paneer Chilly,VEG,315.0,5.0
1,Momo Rice Noodles,Fish Manchurian Gravy,NONVEG,370.0,5.0
2,Momo Rice Noodles,Veg Chilli Garlic Curry with Rice,VEG,250.0,5.0
3,Momo Rice Noodles,Chilli Crispy Fish Dry,NONVEG,370.0,5.0
4,Momo Rice Noodles,Spicy Curd Chicken Lollipop ( 6 pic),NONVEG,331.0,5.0
5,Momo Rice Noodles,Veg Butter Garlic Noodles,VEG,225.0,5.0
6,Beijing Bites,Hunan Tofu,VEG,299.0,5.0
7,Momo Rice Noodles,Pork Schezwan Fried Rice,NONVEG,290.0,5.0
8,Momo Rice Noodles,Seafood Mushroom Clear Soup,NONVEG,195.0,5.0
9,Thai Basil,Stir Fried Chicken Red Curry Paste,NONVEG,480.0,5.0


In [7]:
from typing import Literal, Dict, Any, List


In [8]:
def get_competitors_data(
    q_params: dict,
    output: Literal['dict', 'dataframe'] = 'dict',
    exclude: List[str]=["ids", "city_id"],
) -> List[Dict[str, Any]] | pd.DataFrame:
    graph_query = """
    MATCH (:Area {name: $area})-[:HAS_LOCALITY]->(:Locality)-[:HAS_RESTAURANT]->(r:Restaurant)
    MATCH (r)-[:SERVES_MAIN_CUISINE]->(:MainCuisine {name: $cuisine})
    WHERE r.rating IS NOT NULL AND r.rating >= $min_rating
    RETURN DISTINCT r
    ORDER BY r.rating DESC, r.name ASC
    LIMIT $limit
    """
    result = graph.query(graph_query, q_params).result_set
    rqrd_data = [
        {
            key: ', '.join([str(item) for item in val]) if isinstance(val, list) else val
            for key, val in res[0].properties.items() 
            if key not in exclude
        }
        for res in result
    ]
    # include query time in logs
    return pd.DataFrame(rqrd_data) if output=='dataframe' else rqrd_data

In [9]:
from src.RAG.Config.tool_models import GetCompetitorDataModels

f_params = GetCompetitorDataModels.FunctionParams(
    q_params=GetCompetitorDataModels.QueryParams(
        area='Indiranagar',
        cuisine='Thai',
        limit=200,
    ),
    output='dataframe',
    exclude=['ids','city_id'],
)
cmpt_data = get_competitors_data(**f_params.model_dump())
cmpt_data

,name,city,area,locality,cuisines,rating,address,coords,chain
0,Thai Basil,Bangalore,Indiranagar,Indiranagar,Thai,4.6,840/A/1 Indiranagar 100Ft Road 560038,"12.9806363, 77.6407933",True
1,SHARON TEA STALL,Bangalore,Indiranagar,Indiranagar,"Thai, Bakery",4.5,"Shop No : 1 , Floor : G , SHARON TEA STALL , N...","12.9731234486116, 77.6471368762574",True
2,Xian,Bangalore,Indiranagar,Sarvagna Nagar,"Chinese, Thai",4.5,"# 11,Haudin Road, Diagonally opp God's Gift Ap...","12.961136, 77.635647",False
3,Beijing Bites,Bangalore,Indiranagar,CMH Road,"Chinese, Thai",4.4,"537/2, CMH Road, Indiranagar, Bangalore","12.977945, 77.638814",True
4,Tiger Thai,Bangalore,Indiranagar,100 feet road,"Thai, Pan-Asian",4.4,"No.656, TAIKI, 100 FEET ROAD , INDIRANGAR , BA...","12.9769721, 77.6406962",False
5,Wok In Chow - Pure Vegetarian Asian Street Food,Bangalore,Indiranagar,Appareddipalya,"Chinese, Thai",4.4,"XJCP+4CG, 11th Main Rd, Appareddipalya, Indira...","12.970307402356, 77.636133347885",False
6,China Garten,Bangalore,Indiranagar,Hal 2dd Stage,"Chinese, Thai",4.3,No 82 ground floor vasanthappa garden 13th g m...,"12.965888, 77.639993",False


In [10]:
def get_competitors_menu(
    q_params: dict,
    output: Literal['dict', 'dataframe'] = 'dict',
    exclude: List[str]=["ids", "city_id"],
)-> List[Dict[str, Any]] | pd.DataFrame:
    graph_query = """
    MATCH (:Area {name: $area})-[:HAS_LOCALITY]->(:Locality)-[:HAS_RESTAURANT]->(rstn:Restaurant)
    MATCH (rstn)-[:SERVES_MAIN_CUISINE]->(:MainCuisine {name: $cuisine})
    MATCH (rstn)-[link:HAS_MENU]->(menu:Menu)
    WHERE link.rating IS NOT NULL AND link.rating >= $min_menu_rating

    WITH rstn, menu, link
    ORDER BY rstn.name ASC, link.rating DESC, menu.name ASC

    WITH rstn, collect({
        rstn: properties(rstn),
        menu: properties(menu),
        food: properties(link)
    })[0..20] AS top_menus

    UNWIND top_menus AS merged
    RETURN merged
    LIMIT $limit
    """
    result = graph.query(graph_query, q_params).result_set
    full_data = [
        {
            (
                f"{key1}_{key2}".replace("menu_", "food_", 1)
                if f"{key1}_{key2}".startswith("menu_")
                else f"{key1}_{key2}"
            ): (', '.join(map(str, val)) if isinstance(val, list) else val)
            for key1 in item.keys()
            for key2, val in item[key1].items()
            if f"{key1}_{key2}" not in exclude
        }
        for group in result
        for item in group
    ]
    return pd.DataFrame(full_data) if output=='dataframe' else full_data


In [ ]:
from src.RAG.Config.tool_models import GetCompetitorMenuModels

f_params = GetCompetitorMenuModels.FunctionParams(
    q_params=GetCompetitorMenuModels.QueryParams(
        area='Indiranagar',
        cuisine='Thai',
        min_menu_rating=4.0,
        limit=500,
    ),
    output='dataframe',
    exclude=['rstn_city_id','rstn_ids']
)
data = get_competitors_menu(**f_params.model_dump())
data

,rstn_name,rstn_city,rstn_area,rstn_locality,rstn_cuisines,rstn_rating,rstn_address,rstn_coords,rstn_chain,food_name,food_types,food_price,food_rating
0,China Garten,Bangalore,Indiranagar,Hal 2dd Stage,"Chinese, Thai",4.3,No 82 ground floor vasanthappa garden 13th g m...,"12.965888, 77.639993",False,Beijing Chilli Prawns,NONVEG,260.0,5.0
1,China Garten,Bangalore,Indiranagar,Hal 2dd Stage,"Chinese, Thai",4.3,No 82 ground floor vasanthappa garden 13th g m...,"12.965888, 77.639993",False,Hot Chilli Baby corn,VEG,150.0,5.0
2,China Garten,Bangalore,Indiranagar,Hal 2dd Stage,"Chinese, Thai",4.3,No 82 ground floor vasanthappa garden 13th g m...,"12.965888, 77.639993",False,Mixed Vegetable in Green Chilli Gravy,VEG,180.0,5.0
3,China Garten,Bangalore,Indiranagar,Hal 2dd Stage,"Chinese, Thai",4.3,No 82 ground floor vasanthappa garden 13th g m...,"12.965888, 77.639993",False,Prawns Chinese Chop suey,NONVEG,220.0,5.0
4,China Garten,Bangalore,Indiranagar,Hal 2dd Stage,"Chinese, Thai",4.3,No 82 ground floor vasanthappa garden 13th g m...,"12.965888, 77.639993",False,Spicy Vegetable Curry,VEG,170.0,5.0
